# BR 02 Sanctions/Procurement Crosscheck

Author: Noah Green CPA CFE

Synthetic/redacted Cross-Border Diligence Atlas lab companion. This notebook loads synthetic Brazil entity and sanctions/procurement-style rows, calls the shared `screen_sanctions` helper, and exports a small lead table for human review. Leads are not findings, legal conclusions, jurisdiction-level scores, or suitability recommendations.

## Load Synthetic Inputs

The workflow uses only the lab's synthetic CSVs: `data/synthetic/brazil_entities.csv` and `data/synthetic/brazil_sanctions.csv`.

In [1]:
from csv import DictReader, DictWriter
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from crossborder_dd.brazil_sanctions_screen import screen_sanctions

ENTITIES_PATH = PROJECT_ROOT / "data" / "synthetic" / "brazil_entities.csv"
SANCTIONS_PATH = PROJECT_ROOT / "data" / "synthetic" / "brazil_sanctions.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "redacted_outputs" / "BR_02_sanctions_procurement_crosscheck.csv"


def load_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as handle:
        return list(DictReader(handle))


entities = load_csv(ENTITIES_PATH)
sanctions = load_csv(SANCTIONS_PATH)

{
    "entities_loaded": len(entities),
    "sanctions_rows_loaded": len(sanctions),
}

{'entities_loaded': 2, 'sanctions_rows_loaded': 1}

## Generate Lead Table

Matching logic stays in `crossborder_dd.brazil_sanctions_screen.screen_sanctions`. The notebook adds only workflow context and escalation fields to the resulting lead rows.

In [2]:
raw_leads = screen_sanctions(entities, sanctions)

lead_table = [
    {
        "country": lead["country"],
        "subject": lead["subject"],
        "match_type": lead["match_type"],
        "lead_flag": lead["lead_flag"],
        "source_shape": "synthetic Brazil sanctions/procurement crosscheck",
        "sanctions_escalation": "Confirm against the official sanctions or debarment source before relying on the lead.",
        "procurement_escalation": "Check applicable procurement eligibility, debarment, contract, and source-update rules.",
        "local_counsel_escalation": "Escalate to qualified Brazil local counsel before legal interpretation or transaction advice.",
        "limitation": lead["limitation"],
        "output_status": "lead_only_no_legal_conclusion",
    }
    for lead in raw_leads
]

lead_table

[{'country': 'Brazil',
  'subject': 'Empresa Sintetica Brasil Ltda',
  'match_type': 'exact_identifier',
  'lead_flag': 'synthetic_debarment_lead',
  'source_shape': 'synthetic Brazil sanctions/procurement crosscheck',
  'sanctions_escalation': 'Confirm against the official sanctions or debarment source before relying on the lead.',
  'procurement_escalation': 'Check applicable procurement eligibility, debarment, contract, and source-update rules.',
  'local_counsel_escalation': 'Escalate to qualified Brazil local counsel before legal interpretation or transaction advice.',
  'limitation': 'Lead only. Read the official source and escalate before treating as a finding.',
  'output_status': 'lead_only_no_legal_conclusion'}]

In [3]:
FIELDNAMES = [
    "country",
    "subject",
    "match_type",
    "lead_flag",
    "source_shape",
    "sanctions_escalation",
    "procurement_escalation",
    "local_counsel_escalation",
    "limitation",
    "output_status",
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = DictWriter(handle, fieldnames=FIELDNAMES, lineterminator="\n")
    writer.writeheader()
    writer.writerows(lead_table)

{
    "output_path": str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "rows_written": len(lead_table),
}

{'output_path': 'data/redacted_outputs/BR_02_sanctions_procurement_crosscheck.csv',
 'rows_written': 1}

## What This Proves / What It Cannot Prove

This proves:

- The synthetic entity CSV and synthetic sanctions/procurement-style CSV can be loaded in a reproducible lab workflow.
- `screen_sanctions` can convert an exact identifier or normalized-name match into a review lead.
- The exported table is a redacted lead queue with sanctions, procurement, and local-counsel escalation fields.

This cannot prove:

- That any real Brazil entity is sanctioned, debarred, ineligible, high risk, suitable, or unsuitable.
- That a no-hit clears an entity.
- That source coverage, update cadence, legal effect, appeals/status, or procurement consequence has been verified.
- Any Brazil jurisdiction-level score or legal conclusion.

The workflow treats jurisdiction as source context, not a proxy for risk. It demonstrates a source-literacy pattern for escalating possible sanctions/procurement leads in a specific jurisdiction.

## Required Escalation

Each lead requires:

- Sanctions escalation: read the official sanctions, debarment, or procurement source behind the row; confirm identifiers, aliases, status, scope, and update date.
- Procurement escalation: check procurement eligibility/debarment sources and contract-specific rules before treating a source row as relevant to vendor qualification.
- Local-counsel escalation: qualified Brazil counsel must interpret legal effect, remedies, appeal status, and transaction consequences.

This notebook exports leads only. It does not make findings, reach legal conclusions, or assign jurisdiction-level scores.